In [1]:

import tensorflow as tf
from tensorflow import keras
import numpy as np

#Loading and Preparing the Data
print("Downloading Fashion MNIST dataset...")
# This built-in function fetches the data directly, bypassing the need for Kaggle API keys!
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Images are 0-255 pixels. We scale them to 0.0 - 1.0 for Gradient Descent to work properly.
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# Taking the first 5000 images for validation, and the rest for training
X_valid, X_train = X_train_full[:5000], X_train_full[5000:]
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_valid.shape}")

batch_size = 32

# We shuffle the training data to ensure the model doesn't memorize the order
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(batch_size)

# no need to shuffle the validation and test sets
valid_dataset = tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(batch_size)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size)

print("Data successfully batched and ready for the custom training loop!")

Libraries loaded successfully!
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (55000, 28, 28)
Validation data shape: (5000, 28, 28)
Data successfully batched and ready for the custom training loop!


In [2]:
#Building the Model and Setting up Optimizers

print("\n Building the Model")
model = keras.Sequential([
    # Input layer: flattens the 28x28 pixel image into a 1D line of 784 pixels
    keras.layers.Flatten(input_shape=(28, 28)),

    # LOWER LAYERS: Learning basic edges and shapes
    keras.layers.Dense(128, activation="relu", name="lower_dense_1"),
    keras.layers.Dense(128, activation="relu", name="lower_dense_2"),

    # UPPER LAYERS: Combining shapes into final clothing concepts
    keras.layers.Dense(64, activation="relu", name="upper_dense_1"),

    # OUTPUT LAYER: 10 neurons for the 10 categories of clothing.
    # 'softmax' to turns the math into % (ex 90% sure it's a shirt)
    keras.layers.Dense(10, activation="softmax", name="output_layer")
])

print("Dividing the layers for different learning rates...")
# We grab the weights specifically from the lower layers (indices 1 and 2, because 0 is the Flatten layer)
lower_layers_weights = model.layers[1].trainable_variables + model.layers[2].trainable_variables

# We grab the weights from the upper layers and the output layer (indices 3 and 4)
upper_layers_weights = model.layers[3].trainable_variables + model.layers[4].trainable_variables

print("Initializing Optimizers and Metrics...")

# The Loss Function (The Grader)
# SparseCategoricalCrossentropy is standard for classification with mutually exclusive categories (like clothing types)
loss_fn = keras.losses.SparseCategoricalCrossentropy()

# The Two Optimizers (The Hikers)
# Lower layers= careful learning rate (0.0001)
optimizer_lower = keras.optimizers.Adam(learning_rate=1e-4)
# Upper layers= faster learning rate (0.001)
optimizer_upper = keras.optimizers.Adam(learning_rate=1e-3)

# The Metrics (The Stopwatches to calculate averages)
# We need these to calculate the mean loss and accuracy across all batches in an epoch
train_loss_tracker = keras.metrics.Mean()
train_accuracy_tracker = keras.metrics.SparseCategoricalAccuracy()

val_loss_tracker = keras.metrics.Mean()
val_accuracy_tracker = keras.metrics.SparseCategoricalAccuracy()

print("Model, Optimizers, and Metrics are fully configured!")


--- Building the Model ---
Dividing the layers for different learning rates...
Initializing Optimizers and Metrics...
Model, Optimizers, and Metrics are fully configured!


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
# Alright lets retiring `.fit()` and take manual control of the training process.

import time

# We will run for 3 epochs (full passes through the dataset) for demonstration
epochs = 3

print("\n Starting Custom Training Loop...\n")

for epoch in range(epochs):
    print(f" Epoch {epoch + 1}/{epochs} ")

    # we MUST reset these "stopwatches" at the start of every epoch
    # Otherwise, the averages will bleed into the next epoch.
    train_loss_tracker.reset_state()
    train_accuracy_tracker.reset_state()
    val_loss_tracker.reset_state()
    val_accuracy_tracker.reset_state()

    start_time = time.time()

    #      TRAINING LOOP

    # We iterate through the dataset one batch (32 images) at a time
    for step, (X_batch, y_batch) in enumerate(train_dataset):

        # Turn on the tap aka "Video Recorder" to track the math for Backpropagation
        with tf.GradientTape() as tape:
            # Forward Pass: Make a blind guess
            y_pred = model(X_batch, training=True)

            # Calculate the Error (Loss) for this batch
            loss = loss_fn(y_batch, y_pred)

        # Backpropagation: The Tape calculates the gradients (the slope) for all our specific weights
        all_weights = lower_layers_weights + upper_layers_weights
        gradients = tape.gradient(loss, all_weights)

        # We split the gradients back into two groups to match our optimizers
        num_lower_weights = len(lower_layers_weights)
        gradients_lower = gradients[:num_lower_weights]
        gradients_upper = gradients[num_lower_weights:]

        # Apply the gradients: The hikers take their steps in the loss function valley
        # Lower optimizer takes small steps (1e-4)
        optimizer_lower.apply_gradients(zip(gradients_lower, lower_layers_weights))
        # Upper optimizer takes bigger steps (1e-3)
        optimizer_upper.apply_gradients(zip(gradients_upper, upper_layers_weights))

        # Update the real-time average metrics
        train_loss_tracker.update_state(loss)
        train_accuracy_tracker.update_state(y_batch, y_pred)

        # Print the progress on the SAME line (using '\r' and end="")
        print(f"\r  Iteration: {step + 1} - Mean Loss: {train_loss_tracker.result():.4f} - Mean Accuracy: {train_accuracy_tracker.result():.4f}", end="")

    print() # a new line


    #     VALIDATION LOOP

    # We do NOT need GradientTape here because we are not learning/updating weights.
    # We are just testing the model on unseen data.
    for X_batch_val, y_batch_val in valid_dataset:
        y_pred_val = model(X_batch_val, training=False)
        val_loss = loss_fn(y_batch_val, y_pred_val)

        val_loss_tracker.update_state(val_loss)
        val_accuracy_tracker.update_state(y_batch_val, y_pred_val)

    # Print the final epoch summary
    end_time = time.time()
    print(f"  ↳ End of Epoch {epoch + 1} ({(end_time - start_time):.1f}s) - Val Loss: {val_loss_tracker.result():.4f} - Val Accuracy: {val_accuracy_tracker.result():.4f}\n")

print("Custom Training Loop Complete!")


🚀 Starting Custom Training Loop...

--- Epoch 1/3 ---
  Iteration: 1719 - Mean Loss: 0.5611 - Mean Accuracy: 0.8055
  ↳ End of Epoch 1 (114.1s) - Val Loss: 0.4370 - Val Accuracy: 0.8466

--- Epoch 2/3 ---
  Iteration: 1719 - Mean Loss: 0.3927 - Mean Accuracy: 0.8588
  ↳ End of Epoch 2 (119.4s) - Val Loss: 0.3583 - Val Accuracy: 0.8710

--- Epoch 3/3 ---
  Iteration: 1719 - Mean Loss: 0.3519 - Mean Accuracy: 0.8708
  ↳ End of Epoch 3 (113.6s) - Val Loss: 0.3438 - Val Accuracy: 0.8782

🎉 Custom Training Loop Complete!
